# A8: che tên app và tool khi train (+100 mẫu oos), đo trên app lạ theo bộ chia A4. Khoảng **3,5 đến 4 giờ GPU**.
Ý tưởng, nguồn và các kiểm tra đã làm ở máy local: `docs/Y_TUONG_CHE_TEN.md`. Công thức A4 (`q4b_holdout`: 1180 mẫu + bản ASR `phowl`, 3 epoch) **cộng** `holdout/sft_train_aug.jsonl` (100 mẫu oos) và `holdout/sft_train_mask.jsonl` (1180 bản sao có tên app và tool thay bằng chuỗi ngẫu nhiên). Lúc đo dùng tên thật như lúc thi. Phán quyết theo tiêu chí như A5: **app lạ tăng ít nhất 5 điểm và app đã thấy không giảm quá 3 điểm** so với `q4b_holdout` (41,93 và 66,03).

**Trước khi chạy cell đầu tiên:** đặt notebook **Private**, Internet **Off**, Accelerator **GPU (RTX PRO 6000)**.
**Input cần gắn (4 thứ):**
1. dataset `lifemate-code` **bản MỚI có `src/lifemate/masking.py`** (cell A-1 chặn nếu bản cũ)
2. dataset `train+public_test` (raw)
3. dataset `thuan-qwen3-4b`
4. **output của notebook A4b** (`a4b-seen-reference`): lấy `state/processed` (có `asr_phowl_train.jsonl`, `dev_ids.txt`) và hai file mốc `outputs/unseen_q4b_holdout.jsonl`, `outputs/seen_q4b_holdout.jsonl`

**KHÔNG gắn** `Hello_done`, `merge-lora`, output A4, A5 hay `train-hist` (cùng có `state/processed`, gây khôi phục nhầm). Không cần PhoWhisper.
**Thứ tự cell:** P-1 đến P-3, A-1 (kiểm tra, chạy test che tên), A-2 (dựng dữ liệu, preflight), A-3 (chấm lại hai mốc: phải ra 41,93 và 66,03, lệch thì dừng), A-4 (train, khoảng 3 giờ 10 phút), A-5 (đo), A-6 (phán quyết), A-7 (đóng zip), A-8 (dọn, chạy cuối rồi Save Version).
**Khi xong:** gửi `a8_results.zip` cho Phương qua Drive riêng (có dự đoán trên dữ liệu cuộc thi) và chép dòng phán quyết ở A-6. ĐẠT thì làm bản nộp theo `docs/Y_TUONG_CHE_TEN.md` mục 7; KHÔNG đạt thì giữ bài nộp hiện tại.

In [ ]:
%%bash
# P-1: setup. Dùng kaggle_setup.sh của Phương; chỉ đặt đường dẫn model cho khớp đúng thứ ta gắn:
#  - QWEN = Qwen3-4B (hidden 2560, 36 tầng), tìm theo cấu hình vì còn có Qwen3-8B/Qwen3.5-9B trong input (không lấy "qwen3 đầu tiên")
#  - PHO = đường dẫn giả (A5 không dùng PhoWhisper)
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
find /kaggle/input -maxdepth 8 -name config.json 2>/dev/null > /kaggle/working/config_paths.txt
QW=$(python -X utf8 - <<'EOF'
import json
from pathlib import Path
for line in open("/kaggle/working/config_paths.txt", encoding="utf-8"):
    p = Path(line.strip())
    try:
        c = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue
    t = c.get("text_config", c)
    if c.get("model_type") == "qwen3" and t.get("hidden_size") == 2560 and t.get("num_hidden_layers") == 36:
        print(p.parent)
        break
EOF
)
echo "Qwen3-4B dir: ${QW:-KHÔNG THẤY}"
[ -n "$QW" ] || { echo "ERROR: không thấy Qwen3-4B (hidden 2560, 36 tầng). Gắn dataset thuan-qwen3-4b."; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e 's#^PHO=.*#PHO=/nonexistent/phowhisper#' -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục từ output notebook trước? (checkpoints và outputs)"; ls /kaggle/working/checkpoints | head -20; ls /kaggle/working/outputs | head -12

In [ ]:
%%bash
# P-2: in phiên bản torch/CUDA/GPU, kiểm tra GPU, peft, accelerate và việc transformers nhận qwen3_5
pip uninstall -y torchao 2>&1 | tail -1
python -X utf8 - <<'EOF' | tee /kaggle/working/torch_cuda.txt
import sys
import torch
print(torch.__version__, torch.version.cuda, sys.version)
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__, "| gpus", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("transformers nhận qwen3:", "qwen3" in CONFIG_MAPPING, "| qwen3_5 (Qwen3.5):", "qwen3_5" in CONFIG_MAPPING)
try:
    import accelerate
    print("accelerate", accelerate.__version__)
except ImportError:
    print("THIẾU accelerate: gen_text.py sẽ lỗi ở bước nạp model (device_map). Báo lại, đừng pip install từ mạng.")
if not torch.cuda.is_available():
    raise SystemExit("Phiên này KHÔNG có GPU: bật Accelerator rồi chạy lại")
print("VRAM tổng: %.0f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
EOF

In [ ]:
%%bash
# P-3: ghi sẵn hai đoạn mã phụ (chỉ in số đếm tổng hợp, không in nội dung câu nói hay dự đoán)
mkdir -p /kaggle/working/helpers
cat > /kaggle/working/helpers/a8_verdict.py <<'PYEOF'
"""Phán quyết A8 (che tên app/tool + 100 mẫu oos) theo tiêu chí đặt trước, giống A5 (docs/Y_TUONG_CHE_TEN.md mục 6):
  đạt nếu điểm app LẠ tăng >= 5 điểm (so từng cặp) VÀ điểm app ĐÃ THẤY không giảm quá 3 điểm.
Chỉ in số tổng hợp (không in nội dung dự đoán).
usage: python a8_verdict.py <unseen_ids> <unseen_base.jsonl> <unseen_new.jsonl> <seen_ids> <seen_base.jsonl> <seen_new.jsonl> [<tên mới>]
Chạy từ thư mục gốc repo (cần scripts/compare_runs.py, data/raw/..., data/processed/gold_hash_train.json)."""
import json
import random
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "src"))
import compare_runs as cr  # noqa: E402
from lifemate.metric import load_fuzzy_params  # noqa: E402

u_ids, u_base, u_new, s_ids, s_base, s_new = sys.argv[1:7]
name = sys.argv[7] if len(sys.argv) > 7 else "q4b_holdout_mask"
all_items = [json.loads(line) for line in open(cr.DATA / "train.jsonl", encoding="utf-8")]
gh = json.loads((ROOT / "data" / "processed" / "gold_hash_train.json").read_text(encoding="utf-8"))
fuzzy = load_fuzzy_params(cr.DATA / "train_tools_catalog.json")


def scores(ids_file, sub):
    keep = set(Path(ids_file).read_text().split())
    items = [i for i in all_items if i["id"] in keep]
    s = cr.per_sample(sub, items, gh, fuzzy)
    return {i["id"]: (cr.kind(i["label"]["tool_calls"]), s[i["id"]]) for i in items}


def paired(a, b, n=5000, seed=0):
    ids = sorted(a)
    d = [b[i][1] - a[i][1] for i in ids]
    rng = random.Random(seed)
    boots = sorted(sum(d[rng.randrange(len(d))] for _ in d) / len(d) for _ in range(n))
    return 100 * sum(d) / len(d), 100 * boots[int(0.025 * n)], 100 * boots[int(0.975 * n)], sum(x > 1e-9 for x in d), sum(x < -1e-9 for x in d)


def mean(x):
    return 100 * sum(v for _, v in x.values()) / len(x)


res = {}
for label, ids, base, new in (("app LẠ (eval_ids, 174)", u_ids, u_base, u_new), ("app ĐÃ THẤY (dev_ids, 132)", s_ids, s_base, s_new)):
    a, b = scores(ids, base), scores(ids, new)
    assert set(a) == set(b), "hai bài nộp không cùng tập mẫu"
    d, lo, hi, up, down = paired(a, b)
    res[label] = d
    print(f"== {label}: n={len(a)}")
    print(f"   mốc q4b_holdout {mean(a):.2f}  ->  {name} {mean(b):.2f}   chênh từng cặp {d:+.2f} [{lo:+.2f}; {hi:+.2f}]  ({up} tốt lên, {down} tệ đi, {len(a) - up - down} bằng)")
    for k in ("exec", "ask", "oos"):
        ka = [i for i in a if a[i][0] == k]
        if ka:
            print(f"   {k:5} (n={len(ka):3}): {100 * sum(a[i][1] for i in ka) / len(ka):6.2f} -> {100 * sum(b[i][1] for i in ka) / len(ka):6.2f}")
print("\nTiêu chí đặt trước (như A5): (1) app lạ tăng >= +5,00; (2) app đã thấy không giảm quá 3,00 (tức chênh >= -3,00).")
du, ds = res["app LẠ (eval_ids, 174)"], res["app ĐÃ THẤY (dev_ids, 132)"]
c1, c2 = du >= 5.0, ds >= -3.0
print(f"  (1) app lạ {du:+.2f}: {'ĐẠT' if c1 else 'KHÔNG đạt'}   (2) app đã thấy {ds:+.2f}: {'ĐẠT' if c2 else 'KHÔNG đạt'}")
print(f"  => A8 {'ĐẠT: train bản nộp (docs/Y_TUONG_CHE_TEN.md mục 7), báo Phương' if c1 and c2 else 'KHÔNG ĐẠT: giữ bài nộp hiện tại, báo Phương'}")
print("Lưu ý: dev nhiễu, chênh dưới khoảng 6 điểm khó phân biệt; đọc kèm khoảng tin cậy ở trên (Phương dặn quyết định chính dựa vào tiêu chí 1).")
PYEOF
cat > /kaggle/working/helpers/pack_zip.py <<'PYEOF'
"""Đóng gói file kết quả nhỏ thành một zip kèm MANIFEST sha256 (không đóng gói trọng số).
usage: python pack_zip.py <out.zip> <file> [<file> ...]   (file không tồn tại thì bỏ qua và báo)"""
import hashlib
import sys
import zipfile
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()


out = Path(sys.argv[1])
files = []
for name in sys.argv[2:]:
    p = Path(name)
    if p.is_file():
        files.append(p)
    else:
        print("bỏ qua (không có):", name)
files = sorted(set(files), key=str)
manifest = "".join(f"{sha256(p)}  {p.as_posix()}  ({p.stat().st_size} bytes)\n" for p in files)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.as_posix().lstrip("/"))
    z.writestr("MANIFEST_sha256.txt", manifest)
print(f"{out}: {out.stat().st_size / 1e6:.2f} MB, {len(files)} file")
PYEOF
ls -la /kaggle/working/helpers

In [ ]:
%%bash
# A-1: kiểm tra đủ đồ: MÃ MỚI (có lifemate/masking.py và cờ --aug-mask), dữ liệu từ output A4b, hai file mốc. Thiếu thì DỪNG.
# Rồi cất hai file mốc (dự đoán của q4b_holdout từ A4 và A4b) sang thư mục riêng và dọn bản sao không dùng.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
MISSING=0
need() { if [ -s "$1" ]; then echo "ok     $1"; else echo "THIẾU  $1"; MISSING=1; fi; }
need scripts/build_sft.py
need scripts/train_sft.py
need scripts/preflight_train.py
need scripts/compare_runs.py
need scripts/infer_multi_gpu.sh
need src/lifemate/masking.py
need data/processed/asr_phowl_train.jsonl
need data/processed/dev_ids.txt
need data/processed/gold_hash_train.json
need outputs/unseen_q4b_holdout.jsonl
need outputs/seen_q4b_holdout.jsonl
if grep -q -- "--aug-mask" scripts/build_sft.py 2>/dev/null; then
  echo "ok     build_sft.py có --aug-mask (mã mới)"
else
  echo "THIẾU  build_sft.py chưa có --aug-mask: upload lifemate-code MỚI (có masking.py)"; MISSING=1
fi
if [ "$MISSING" = 1 ]; then echo "-> Gắn output notebook A4b (có state/, outputs/unseen_*, outputs/seen_*) và lifemate-code mới."; exit 1; fi
n=$(wc -l < data/processed/dev_ids.txt)
[ "$n" = 155 ] || { echo "dev_ids.txt có $n dòng, phải là 155"; exit 1; }
echo "dev_ids.txt: 155 id (đúng)"
echo "-- test của phần che tên (nhanh, không cần GPU)"
python -X utf8 -m pytest -q tests/test_masking.py 2>&1 | tail -3
echo "-- cất hai file mốc, dọn bản sao không dùng"
mkdir -p /kaggle/working/baseline
cp outputs/unseen_q4b_holdout.jsonl outputs/seen_q4b_holdout.jsonl /kaggle/working/baseline/
rm -rf /kaggle/working/outputs/* /kaggle/working/checkpoints/*
mkdir -p outputs/logs
rm -rf data/processed/holdout
ls /kaggle/working/baseline; df -h /kaggle/working | tail -1

In [ ]:
%%bash
# A-2: dựng dữ liệu bộ chia A4 + 100 mẫu oos + 1180 mẫu che tên (ghi vào data/processed/holdout/). Kỳ vọng: 1180 train, 174 eval, 132 dev,
# "augmentation -> sft_train_aug.jsonl: 100 samples" và "name masking -> sft_train_mask.jsonl: 1180 samples". Rồi preflight (0 bước quá 5120).
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
HOLD=flights,bike_rental,package_tracking,shopping_list,podcast_radio,loyalty
python -X utf8 scripts/build_sft.py --asr-tag phowl --hold-apps $HOLD --aug-oos 100 --aug-mask 1180 2>&1 | cut -c1-400 | tee outputs/logs/a8_build.txt
H=data/processed/holdout
wc -l $H/sft_train.jsonl $H/sft_train_asr.jsonl $H/sft_train_aug.jsonl $H/sft_train_mask.jsonl $H/eval_ids.txt $H/dev_ids.txt
if [ "$(wc -l < $H/sft_train.jsonl)" != 1180 ] || [ "$(wc -l < $H/eval_ids.txt)" != 174 ] || [ "$(wc -l < $H/dev_ids.txt)" != 132 ]; then
  echo "SỐ MẪU KHÁC KỲ VỌNG (1180 train / 174 eval / 132 dev của bộ chia A4): dừng, gửi output này cho Phương"; exit 1
fi
[ "$(wc -l < $H/sft_train_aug.jsonl)" = 100 ] || { echo "sft_train_aug.jsonl không đủ 100 mẫu oos: dừng"; exit 1; }
[ "$(wc -l < $H/sft_train_mask.jsonl)" = 1180 ] || { echo "sft_train_mask.jsonl không đủ 1180 mẫu: dừng"; exit 1; }
sha256sum $H/sft_train_mask.jsonl $H/sft_train_aug.jsonl | tee -a outputs/logs/a8_build.txt
python -X utf8 scripts/preflight_train.py --data $H/sft_train.jsonl $H/sft_train_asr.jsonl $H/sft_train_aug.jsonl $H/sft_train_mask.jsonl \
  --dev-ids $H/dev_ids.txt --hold-apps $HOLD --max-len 5120 --tokenizer $QWEN 2>&1 | tee outputs/logs/a8_preflight.txt
if grep -q "total training steps per epoch: 11437" outputs/logs/a8_preflight.txt; then
  echo "preflight: 11437 bước mỗi epoch (đúng như đo ở máy local)"
else
  echo "CHÚ Ý: số bước mỗi epoch khác 11437 của máy local; vẫn train được nếu preflight OK, ghi lại để báo"
fi

In [ ]:
%%bash
# A-3: chấm lại HAI FILE MỐC (q4b_holdout từ A4 và A4b) trên đúng tệp id vừa dựng: phải ra 41,93 (app lạ, 174) và 66,03 (app đã thấy, 132).
# Lệch thì DỪNG (nghĩa là bộ chia không trùng A4, phán quyết sẽ vô nghĩa).
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
B=/kaggle/working/baseline
python -X utf8 /kaggle/working/helpers/a8_verdict.py $H/eval_ids.txt $B/unseen_q4b_holdout.jsonl $B/unseen_q4b_holdout.jsonl \
  $H/dev_ids.txt $B/seen_q4b_holdout.jsonl $B/seen_q4b_holdout.jsonl baseline 2>&1 | tee outputs/logs/a8_baseline_check.txt | grep "mốc q4b_holdout"
if grep -q "mốc q4b_holdout 41.93" outputs/logs/a8_baseline_check.txt && grep -q "mốc q4b_holdout 66.03" outputs/logs/a8_baseline_check.txt; then
  echo "MỐC KHỚP (41,93 và 66,03): được train"
else
  echo "MỐC KHÔNG KHỚP: dừng, báo Phương (không train)"; exit 1
fi

In [ ]:
%%bash
# A-4: TRAIN q4b_holdout_mask = công thức A4 (q4b_holdout: sft_train + sft_train_asr, 3 epoch) + sft_train_aug (100 oos) + sft_train_mask (1180 mẫu che tên).
# Khoảng 3 giờ 5 phút đến 3 giờ 20 phút (11437 bước mỗi epoch; A5 10549 bước mất 2 giờ 52 phút). Chạy bằng Save Version thì log chỉ hiện khi cell xong.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
[ -s $H/sft_train_mask.jsonl ] || { echo "THIẾU $H/sft_train_mask.jsonl: chạy A-2 trước"; exit 1; }
START=$(date +%s)
python -X utf8 scripts/train_sft.py --model $QWEN --data $H/sft_train.jsonl $H/sft_train_asr.jsonl $H/sft_train_aug.jsonl $H/sft_train_mask.jsonl \
  --out checkpoints/q4b_holdout_mask --epochs 3 --max-len 5120 --global-batch 16 --save-limit 2 2>&1 | tee outputs/logs/a8_q4b_holdout_mask_train.log | tail -5
echo "thời gian train: $(( $(date +%s) - START )) s" | tee outputs/logs/a8_train_time.txt
ls checkpoints/q4b_holdout_mask
sha256sum checkpoints/q4b_holdout_mask/adapter_model.safetensors | tee outputs/logs/a8_adapter_sha256.txt

In [ ]:
%%bash
# A-5: đo q4b_holdout_mask trên app LẠ (174) và app ĐÃ THẤY (132), transcript sạch, luật bật, đúng lệnh như A4, A4b và A5. Khoảng 15 đến 20 phút.
# Lúc đo dùng TÊN THẬT (suy luận không che gì), đúng như lúc thi.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
bash scripts/infer_multi_gpu.sh train outputs/unseen_q4b_holdout_mask.jsonl --ids $H/eval_ids.txt --backend hf --model $QWEN --adapter checkpoints/q4b_holdout_mask --text clean
python -X utf8 scripts/eval.py --sub outputs/unseen_q4b_holdout_mask.jsonl --ids $H/eval_ids.txt | tee outputs/logs/a8_unseen_eval.txt
tail -qn 1 outputs/unseen_q4b_holdout_mask.part*.log | tee outputs/logs/a8_unseen_timing.txt
bash scripts/infer_multi_gpu.sh train outputs/seen_q4b_holdout_mask.jsonl --ids $H/dev_ids.txt --backend hf --model $QWEN --adapter checkpoints/q4b_holdout_mask --text clean
python -X utf8 scripts/eval.py --sub outputs/seen_q4b_holdout_mask.jsonl --ids $H/dev_ids.txt | tee outputs/logs/a8_seen_eval.txt
tail -qn 1 outputs/seen_q4b_holdout_mask.part*.log | tee outputs/logs/a8_seen_timing.txt

In [ ]:
%%bash
# A-6: PHÁN QUYẾT theo tiêu chí đặt trước (như A5: app lạ +5 trở lên, app đã thấy không giảm quá 3) và so từng cặp với q4b_holdout bằng compare_runs.py
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
B=/kaggle/working/baseline
{
python -X utf8 /kaggle/working/helpers/a8_verdict.py $H/eval_ids.txt $B/unseen_q4b_holdout.jsonl outputs/unseen_q4b_holdout_mask.jsonl \
  $H/dev_ids.txt $B/seen_q4b_holdout.jsonl outputs/seen_q4b_holdout_mask.jsonl q4b_holdout_mask
echo
echo "################ compare_runs: app LẠ (q4b_holdout -> q4b_holdout_mask)"
python -X utf8 scripts/compare_runs.py --a $B/unseen_q4b_holdout.jsonl --b outputs/unseen_q4b_holdout_mask.jsonl --ids $H/eval_ids.txt --names q4b_holdout,q4b_holdout_mask
echo
echo "################ compare_runs: app ĐÃ THẤY (q4b_holdout -> q4b_holdout_mask)"
python -X utf8 scripts/compare_runs.py --a $B/seen_q4b_holdout.jsonl --b outputs/seen_q4b_holdout_mask.jsonl --ids $H/dev_ids.txt --names q4b_holdout,q4b_holdout_mask
} 2>&1 | tee outputs/logs/a8_verdict.txt

In [ ]:
%%bash
# A-7: gói kết quả nhỏ: a8_results.zip (dự đoán, điểm, phán quyết, log, sha256 adapter, môi trường). Dự đoán là dữ liệu cuộc thi: chỉ gửi qua Drive riêng.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 /kaggle/working/helpers/pack_zip.py /kaggle/working/a8_results.zip \
  outputs/unseen_q4b_holdout_mask.jsonl outputs/seen_q4b_holdout_mask.jsonl outputs/logs/a8_*.txt outputs/logs/a8_*.log \
  data/processed/holdout/MANIFEST.build_sft.json /kaggle/working/torch_cuda.txt
du -sh /kaggle/working/checkpoints /kaggle/working/outputs; df -h /kaggle/working | tail -1

In [ ]:
%%bash
# A-8 (CUỐI CÙNG): dọn (xóa lifemate/, checkpoint trung gian), giữ adapter cuối, dự đoán, zip và log. Rồi Save Version.
# KHÔNG chạy kaggle_save_state.sh: output này không cần state/ (nếu ĐẠT, bản nộp train trên toàn bộ train từ output Hello_done).
cd /kaggle/working
rm -rf checkpoints/q4b_holdout_mask/checkpoint-*
rm -rf lifemate helpers setup_patched.sh config_paths.txt state
du -sh /kaggle/working/* 2>/dev/null
ls /kaggle/working /kaggle/working/checkpoints
echo "xong. Bây giờ Save Version (private). Lấy a8_results.zip ở tab Output."